# Stage 3 — Symptom Tree

Builds a **hierarchical symptom tree** per admission from the clinical note + Stage 2 extraction.

**LLM backend:** `LLM_PROVIDER` in `pipeline_config.py` (`ollama` or `api`)

**Next:** Run `stage_04_export_patient_records.ipynb`


## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ollama_agents.py").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from cohort_selection import load_cohort
from llm_client import LLMNotAvailableError, check_llm
from ollama_agents import aggregate_patient_symptom_tree_llm, symptom_tree_agent
from pipeline_config import get_llm_config, print_pipeline_banner
from pipeline_io import load_ie_results, save_symptom_tree_results

print_pipeline_banner()
LLM_CONFIG = get_llm_config(for_symptom_tree=True)

ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready — {LLM_CONFIG.provider}: {model_info}")

cohort_df = load_cohort()
ie_df = load_ie_results()
merged = cohort_df.merge(
    ie_df[["hadm_id", "extracted", "extraction_method"]],
    on="hadm_id",
    how="inner",
)
print(f"Admissions to process: {len(merged)}")


## Build Admission-Level Symptom Trees

In [ ]:
trees = []

for _, row in merged.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    print(f"Symptom tree hadm_id={hadm_id}...")
    tree = symptom_tree_agent(
        clinical_note=row["clinical_note"],
        extracted=row["extracted"],
        admission_id=hadm_id,
        patient_id=patient_id,
        config=LLM_CONFIG,
    )
    trees.append({
        **{k: row[k] for k in [
            "patient_id", "admission_id", "subject_id", "hadm_id",
            "primary_icd_code", "primary_dx_title",
            "ground_truth_icd10", "ground_truth_dx_titles", "n_diagnoses",
            "extraction_method", "extracted",
        ]},
        "symptom_tree_method": tree.get("_method", "unknown"),
        "symptom_tree": tree,
        "branch_count": len(tree.get("branches", [])),
    })

results_df = pd.DataFrame(trees)
results_df[["patient_id", "admission_id", "symptom_tree_method", "branch_count"]].head()


## Aggregate Patient-Level Trees

In [ ]:
patient_symptom_trees = {}

for patient_id, grp in results_df.groupby("patient_id"):
    admission_trees = grp["symptom_tree"].tolist()
    print(f"Aggregate patient {patient_id} ({len(admission_trees)} admissions)...")
    patient_symptom_trees[patient_id] = aggregate_patient_symptom_tree_llm(
        admission_trees, patient_id, config=LLM_CONFIG
    )

len(patient_symptom_trees)


## Save Results

In [ ]:
out = save_symptom_tree_results(results_df, patient_symptom_trees)
print(f"Saved → {out}")
